# MLP probe #1 — DL sanity check on the LightGBM feature bank

**Question:** is an MLP head even competitive with LightGBM on the *identical* 1074-feature matrix?

- MLP ~= GBT -> a sequence encoder is worth building (DL can learn from this data scale).
- MLP << GBT -> DL fights the data-scale battle; pivot to blend-only.

Also reports **rank-corr** between MLP and GBT holdout preds. Low corr = orthogonal view worth blending even if MLP loses head-to-head.

Baseline: fresh 3-seed LightGBM bag retrained on the same 1074 features (`params.json`). Holdout = 2000 val series (`feat_final_holdout.parquet`).

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys, time, json
import numpy as np, pandas as pd
import torch, torch.nn as nn

HERE = os.path.abspath("")
EXPERIMENTS = os.path.abspath(os.path.join(HERE, "..", ".."))
sys.path.insert(0, EXPERIMENTS)
import sb_common as C

FEAT_TRAIN = os.path.join(EXPERIMENTS, "feat_final_train.parquet")
FEAT_HOLD  = os.path.join(EXPERIMENTS, "feat_final_holdout.parquet")
LGB_DIR    = os.path.join(EXPERIMENTS, "models", "bank_lgbm")

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
SEED = 0
print("device", DEVICE)

device mps


## Load features, labels, per-series weights

Train = the `tr_ids` rows of the full bank. Holdout = the 2000 `va_ids` series bank. Weights make each *series* count equally (matches the GBT reproducer).

In [2]:
tr_ids = np.load(os.path.join(EXPERIMENTS, "tr_ids.npy"), allow_pickle=True)
y_full = C.load_y()

F = pd.read_parquet(FEAT_TRAIN)
cols = list(F.columns)
ids = F.index.get_level_values("id").to_numpy()
mtr = np.isin(ids, tr_ids)
Xtr = F.to_numpy(np.float32)[mtr]
ytr = y_full.reindex(F.index).to_numpy().astype(np.float32)[mtr]
wtr = C.eq_series_weight(F.index[mtr]).astype(np.float32)
del F

Fh = pd.read_parquet(FEAT_HOLD)[cols]
Xh_raw = Fh.to_numpy(np.float32)
h_idx = Fh.index
yh = pd.Series(y_full.reindex(h_idx).to_numpy().astype(np.int8), index=h_idx)
del Fh

print(f"train {Xtr.shape}  pos {ytr.mean():.3f} | holdout {Xh_raw.shape}")

train (174299, 1072)  pos 0.123 | holdout (43679, 1072)


In [3]:
# standardize on train stats (MLP needs it; GBT does not)
mu = Xtr.mean(0)
sd = Xtr.std(0); sd[sd < 1e-6] = 1.0
Xtr_s = (Xtr - mu) / sd
Xh_s  = (Xh_raw - mu) / sd
np.clip(Xtr_s, -10, 10, out=Xtr_s)   # tame residual outliers post-zscore
np.clip(Xh_s,  -10, 10, out=Xh_s)

array([[ 0.33734235,  0.33734235,  1.011184  , ..., -0.05791218,
        -1.321765  ,  0.        ],
       [-0.9464904 , -0.9464904 , -0.98876816, ..., -0.05791218,
        -1.321765  ,  0.        ],
       [-1.0455711 , -1.0455711 , -0.98876816, ..., -0.05791218,
        -1.321765  ,  0.        ],
       ...,
       [ 0.28932023,  0.28932023,  1.011184  , ..., -0.0822123 ,
        -0.22381586,  0.        ],
       [-0.37685716, -0.37685716, -0.98876816, ..., -0.0822123 ,
        -0.22381586,  0.        ],
       [-0.33040377, -0.33040377, -0.98876816, ..., -0.0822123 ,
        -0.22381586,  0.        ]], shape=(43679, 1072), dtype=float32)

## GBT baseline — fresh 3-seed LightGBM on the *same* 1074 features

The saved `bank_lgbm/model_seed*.txt` are stale (105-feature era) and can't map onto the current 1074 matrix, so we retrain here from `params.json`. Same inputs as the MLP = clean apples-to-apples. Reproduces the ~0.606 holdout.

In [4]:
import lightgbm as lgb
PARAMS = json.load(open(os.path.join(LGB_DIR, "params.json")))
gbt_preds = []
for s in (0, 1, 2):
    m = lgb.LGBMClassifier(random_state=s, **PARAMS).fit(Xtr, ytr.astype(np.int8), sample_weight=wtr)
    gbt_preds.append(C.rank01(m.predict_proba(Xh_raw)[:, 1]))
    print(f"  seed{s} fitted")
gbt_pred = pd.Series(np.mean(gbt_preds, axis=0), index=h_idx)
gbt_auc = C.ts_auc(gbt_pred, yh)
print(f"GBT holdout TS-AUC {gbt_auc:.4f}")

/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  seed0 fitted


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  seed1 fitted


/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  seed2 fitted


GBT holdout TS-AUC 0.5793


## MLP: 1074 -> [512, 256, 128] -> 1, BatchNorm + Dropout, weighted BCE

In [5]:
class MLP(nn.Module):
    def __init__(self, d_in, hidden=(512, 256, 128), p=0.3):
        super().__init__()
        layers, d = [], d_in
        for h in hidden:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(p)]
            d = h
        layers += [nn.Linear(d, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

In [6]:
def train_mlp(Xtr, ytr, wtr, Xh, h_idx, yh, epochs=60, bs=4096, lr=1e-3, wd=1e-5):
    torch.manual_seed(SEED); np.random.seed(SEED)
    Xtr_t = torch.from_numpy(Xtr); ytr_t = torch.from_numpy(ytr); wtr_t = torch.from_numpy(wtr)
    Xh_t = torch.from_numpy(Xh).to(DEVICE)
    model = MLP(Xtr.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=epochs, steps_per_epoch=(len(Xtr)+bs-1)//bs)
    lossf = nn.BCEWithLogitsLoss(reduction="none")
    n = len(Xtr); best = (0.0, -1); best_pred = None; hist = []; t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            xb = Xtr_t[idx].to(DEVICE); yb = ytr_t[idx].to(DEVICE); wb = wtr_t[idx].to(DEVICE)
            opt.zero_grad()
            loss = (lossf(model(xb), yb) * wb).mean()
            loss.backward(); opt.step(); sched.step()
        model.eval()
        with torch.no_grad():
            ph = torch.sigmoid(model(Xh_t)).cpu().numpy()
        auc = C.ts_auc(pd.Series(ph, index=h_idx), yh); hist.append(auc)
        if auc > best[0]:
            best = (auc, ep); best_pred = pd.Series(ph, index=h_idx)
        if ep % 5 == 0 or ep == epochs-1:
            print(f"  ep{ep:02d}  loss {loss.item():.4f}  holdout {auc:.4f}"
                  f"  (best {best[0]:.4f}@{best[1]})  {time.time()-t0:.0f}s")
    return best, best_pred, hist

(best_auc, best_ep), mlp_pred, hist = train_mlp(Xtr_s, ytr, wtr, Xh_s, h_idx, yh)

  ep00  loss 0.6257  holdout 0.5565  (best 0.5565@0)  7s


  ep05  loss 0.3531  holdout 0.5561  (best 0.5579@3)  11s


  ep10  loss 0.2969  holdout 0.5645  (best 0.5660@9)  17s


  ep15  loss 0.2663  holdout 0.5495  (best 0.5669@13)  22s


  ep20  loss 0.2458  holdout 0.5429  (best 0.5669@13)  28s


...

  ep40  loss 0.1263  holdout 0.5443  (best 0.5669@13)  50s


  ep45  loss 0.1031  holdout 0.5489  (best 0.5669@13)  55s


  ep50  loss 0.0978  holdout 0.5458  (best 0.5669@13)  60s


  ep55  loss 0.0867  holdout 0.5486  (best 0.5669@13)  65s


  ep59  loss 0.0776  holdout 0.5480  (best 0.5669@13)  69s


## Verdict — MLP vs GBT, decorrelation, naive blend

In [7]:
corr = np.corrcoef(C.rank01(mlp_pred.to_numpy()), gbt_pred.to_numpy())[0, 1]
blend = pd.Series(0.5*C.rank01(mlp_pred.to_numpy()) + 0.5*gbt_pred.to_numpy(), index=h_idx)
blend_auc = C.ts_auc(blend, yh)
print(f"MLP   holdout TS-AUC {best_auc:.4f} (epoch {best_ep})")
print(f"GBT   holdout TS-AUC {gbt_auc:.4f}")
print(f"gap   {best_auc-gbt_auc:+.4f}   rank-corr {corr:.3f}")
print(f"50/50 blend        {blend_auc:.4f}  ({blend_auc-gbt_auc:+.4f} vs GBT)")

mlp_pred.to_frame("mlp").to_parquet(os.path.join(HERE, "mlp_holdout_pred.parquet"))
json.dump({"mlp_auc": float(best_auc), "mlp_epoch": int(best_ep),
           "gbt_auc": float(gbt_auc), "corr": float(corr),
           "blend_auc": float(blend_auc)},
          open(os.path.join(HERE, "result.json"), "w"), indent=2)

MLP   holdout TS-AUC 0.5669 (epoch 13)
GBT   holdout TS-AUC 0.5793
gap   -0.0124   rank-corr 0.872
50/50 blend        0.5791  (-0.0002 vs GBT)
